# Notebook 04 — Win Type Classification (2025 BDB)

**Inputs:** `outputs/blocker_metrics.csv`, `players.csv` (weight filter)  
**Outputs:** `outputs/win_types.csv`, `outputs/player_win_type_summary.csv`  
**Goal:** Label each blocked rush attempt with a win type (speed / power / counter / loss) using a priority cascade, then compute a context-adjusted pressure rate per player that accounts for how hard it was to rush (single-blocked vs. double-teamed).

**Position filter:** `weight < 300 lbs` applied at load time to remove interior DL carrying DE/OLB roster labels. Confirmed contaminating players (Poona Ford 320 lbs, Kenny Clark 315 lbs, Christian Wilkins 315 lbs) are excluded. Lighter interior-alignment players remain a known limitation — pre-snap y-coordinate alignment is the principled V2 fix.

## Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
OUT_DIR = PROJECT_ROOT / "outputs"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok = True)

## Load Inputs

Load `blocker_metrics.csv` (23,345 rush attempts), then apply two filters:
1. **Weight filter** (`weight < 300 lbs`) — removes interior DL carrying DE/OLB roster labels
2. **Unblocked filter** — drops rushes with no assigned blocker (win type classification requires a blocker to measure against)

Also merges `peakBurstSpeed` from `getoff_metrics.csv`. After both filters: **20,080 blocked rush attempts** remain.

In [2]:
blocker_metrics = pd.read_csv(OUT_DIR / "blocker_metrics.csv")

print(blocker_metrics.shape)
print(blocker_metrics.columns.tolist())

players = pd.read_csv(DATA_DIR / "players.csv")[['nflId', 'weight']]
blocker_metrics = blocker_metrics.merge(players, on='nflId', how='left')
blocker_metrics = blocker_metrics[blocker_metrics['weight'] < 300]
print("After weight filter:", blocker_metrics.shape)
print("Max weight:", blocker_metrics['weight'].max())

blocker_metrics = blocker_metrics[blocker_metrics['blocking_context'] != 'unblocked']
print(blocker_metrics.shape)

getoff = pd.read_csv(OUT_DIR / "getoff_metrics.csv")[['gameId', 'playId', 'nflId', 'peakBurstSpeed']]
blocker_metrics = blocker_metrics.merge(getoff, on=['gameId', 'playId', 'nflId'], how='left')

print(blocker_metrics.shape)
print(blocker_metrics['peakBurstSpeed'].isnull().sum())

(23345, 14)
['gameId', 'playId', 'nflId', 'displayName', 'position', 'teamAbbr', 'blocking_context', 'primary_blocker_nflId', 'causedPressure', 'separation_at_15', 'separation_at_25', 'blocker_displacement', 'rusher_dir_change', 'hip_turn_delta']
After weight filter: (22071, 15)
Max weight: 297
(20080, 15)
(20080, 16)
17


## Win Type Cascade

Priority order — each play gets exactly one label:

| Priority | Label | Condition |
|---|---|---|
| 1 | `counter` | `rusher_dir_change > 45°` **and** `separation_at_25 > 0` |
| 2 | `power` | `blocker_displacement >= 7.5` |
| 3 | `speed` | `separation_at_15 > -2.0` **and** `blocker_displacement < 7.5` **and** `rusher_dir_change <= 45°` |
| 4 | `loss` | none of the above |

In [3]:
speed = (blocker_metrics['separation_at_15'] > -2.0) & \
        (blocker_metrics['blocker_displacement'] < 7.5) & \
        (blocker_metrics['rusher_dir_change'] <= 45)

power = (blocker_metrics['blocker_displacement'] >= 7.5)

counter = (blocker_metrics['rusher_dir_change'] > 45) & (blocker_metrics['separation_at_25'] > 0)

blocker_metrics['win_type'] = np.select(
    [counter, power, speed],
    ['Counter', 'Power', 'Speed'],
    default='Loss'
)

print(blocker_metrics['win_type'].value_counts())

Loss       10809
Speed       8119
Power        593
Counter      559
Name: win_type, dtype: int64


## Context-Adjusted Pressure Rate

Not all rush attempts are equal — it is harder to generate pressure when double-teamed than when single-blocked. This metric gives each play a score based on how difficult the blocking situation was.

- A pressure play against a double team scores **higher** than a pressure play against one blocker
- A play with no pressure scores **0** regardless of context
- Averaged across all of a player's rushes: **1.0 = league average**, **1.5 = 50% better than average** for the blocking they faced

In [4]:
context_baseline = blocker_metrics.groupby('blocking_context')['causedPressure'].mean()
context_baseline = context_baseline.replace(0, float('nan'))

blocker_metrics['context_contribution'] = blocker_metrics['causedPressure'] / blocker_metrics['blocking_context'].map(context_baseline)

print(context_baseline)
print(blocker_metrics['context_contribution'].describe())

blocking_context
double_teamed     0.070642
single_blocked    0.117881
Name: causedPressure, dtype: float64
count    20080.000000
mean         1.000000
std          2.982489
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         14.155807
Name: context_contribution, dtype: float64


## Player Summary

Aggregate to one row per player. Include: total rushes, win type counts, adjusted pressure rate, double-team rate.  
Minimum sample filter: **≥ 75 rush attempts** for overall rankings.

In [5]:
blocker_metrics['is_speed'] = (blocker_metrics['win_type'] == 'Speed')
blocker_metrics['is_power'] = (blocker_metrics['win_type'] == 'Power')
blocker_metrics['is_counter'] = (blocker_metrics['win_type'] == 'Counter')
blocker_metrics['is_loss'] = (blocker_metrics['win_type'] == 'Loss')
blocker_metrics['is_double_teamed'] = (blocker_metrics['blocking_context'] == 'double_teamed')

player_summary = blocker_metrics.groupby(['nflId', 'displayName', 'position', 'teamAbbr']).agg(
    total_rushes = ('win_type', 'count'),
    speed_wins = ('is_speed', 'sum'),
    power_wins = ('is_power', 'sum'),
    counter_wins = ('is_counter', 'sum'),
    losses = ('is_loss', 'sum'),
    adjusted_pressure_rate = ('context_contribution', 'mean'),
    double_team_rate = ('is_double_teamed', 'mean')
).reset_index()

player_summary = player_summary[player_summary['total_rushes'] >= 75]

print(player_summary.shape)
print(player_summary.sort_values('adjusted_pressure_rate', ascending=False).head(10))

(108, 11)
     nflId       displayName position teamAbbr  total_rushes  speed_wins  \
58   44813     Myles Garrett       DE      CLE           190          44   
42   42465   Za'Darius Smith       DE      MIN           196          64   
9    37145    Justin Houston      OLB      BAL           118          28   
70   44915  Trey Hendrickson       DE      CIN           192          67   
108  47785         Nick Bosa       DE       SF           141          35   
172  53441     Micah Parsons      OLB      DAL           171          49   
174  53447   Jaelan Phillips      OLB      MIA           195          68   
128  47944   Charles Omenihu       DE       SF           136          78   
0    35452    Brandon Graham       DE      PHI           119          25   
38   42403     Randy Gregory      OLB      DEN            75          21   

     power_wins  counter_wins  losses  adjusted_pressure_rate  \
58            9             8     129                2.099548   
42            3        

## Validation

Three sanity checks:
1. **Blocking context × win type crosstab** — confirms all four win types appear in both blocking contexts (no empty cells)
2. **Pressure rate by win type** — power wins should have the highest pressure rate; loss should not be included (a lost rep contributes near-zero pressure by definition)
3. **Pressure rate by blocking context** — single-blocked should be higher than double-teamed (harder to generate pressure when facing two blockers)

In [6]:
print(pd.crosstab(blocker_metrics['blocking_context'], blocker_metrics['win_type']))

print(blocker_metrics[blocker_metrics['win_type'] != 'Loss']\
    .groupby('win_type')['causedPressure'].mean().round(3))

print(blocker_metrics.groupby('blocking_context')['causedPressure'].mean().round(3))

win_type          Counter  Loss  Power  Speed
blocking_context                             
double_teamed         221  1891     99   2786
single_blocked        338  8918    494   5333
win_type
Counter    0.048
Power      0.130
Speed      0.084
Name: causedPressure, dtype: float64
blocking_context
double_teamed     0.071
single_blocked    0.118
Name: causedPressure, dtype: float64


## Save Output

In [7]:
blocker_metrics.to_csv(OUT_DIR / "win_types.csv", index=False)
player_summary.to_csv(OUT_DIR / "player_win_type_summary.csv", index=False)

print("Shape:", blocker_metrics.shape)
print("Columns:", blocker_metrics.columns.tolist())
print("Unique rushers:", blocker_metrics['nflId'].nunique())

Shape: (20080, 23)
Columns: ['gameId', 'playId', 'nflId', 'displayName', 'position', 'teamAbbr', 'blocking_context', 'primary_blocker_nflId', 'causedPressure', 'separation_at_15', 'separation_at_25', 'blocker_displacement', 'rusher_dir_change', 'hip_turn_delta', 'weight', 'peakBurstSpeed', 'win_type', 'context_contribution', 'is_speed', 'is_power', 'is_counter', 'is_loss', 'is_double_teamed']
Unique rushers: 239
